In [0]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

df_pd = spark.table("silver.sop_forecast.vendas_limpo").toPandas()

resultados = []
for loja in sorted(df_pd["loja_id"].unique()):
    sub = df_pd[df_pd["loja_id"] == loja]
    mensal = sub.groupby(["ano_mes", "item_id"])["vendas"].sum().reset_index()
    pivot = mensal.pivot(index="ano_mes", columns="item_id", values="vendas")

    def indice_sazonal(serie):
        tmp = pd.DataFrame({"v": serie})
        tmp["mes"] = pd.to_datetime(tmp.index).month
        tmp["ano"] = pd.to_datetime(tmp.index).year
        tmp["idx"] = tmp["v"] / tmp.groupby("ano")["v"].transform("mean")
        return tmp.groupby("mes")["idx"].mean()

    indices = pd.DataFrame({c: indice_sazonal(pivot[c].dropna()) for c in pivot.columns})
    X = MinMaxScaler().fit_transform(indices).T
    labels = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X)

    for item, cluster in zip(indices.columns, labels):
        resultados.append({"loja_id": int(loja), "item_id": int(item), "cluster_id": int(cluster)})

df_clusters = spark.createDataFrame(pd.DataFrame(resultados))

(df_clusters.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.sop_forecast.clusters_sku"))

display(df_clusters.groupBy("loja_id", "cluster_id").count().orderBy("loja_id", "cluster_id"))